# Ensembl genes table extraction EDA

This notebook is useful for development as well as exploratory data analysis on the extracted tables.
It is currently automically executed and saved as part of exports using `papermill`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from bioregistry import normalize_prefix

from ensembl_genes import ensembl_genes

In [3]:
# parameters cell
species = "human"
release = "111"

In [4]:
# Parameters
species = "mouse"
release = "116"


In [5]:
ensg = ensembl_genes.Ensembl_Gene_Queries(release=release, species=species)
ensg.connection_url

'mysql+mysqlconnector://anonymous@ensembldb.ensembl.org:3306/mus_musculus_core_116_39'

In [6]:
database = ensg.database
database

'mus_musculus_core_116_39'

## Extract data

## gene attrib counts

In [7]:
ensg.run_query("gene_attrib_counts").head(15)

,attrib_type_id,code,name,description,attrib_type_count,attrib_type_examples
0,142,GeneGC,Gene GC,Percentage GC content for this gene,78348,"44.76, 43.96, 42.75, 45.47, 49.48, 43.52, 52.5..."
1,4,name,Name,Alternative/long name,72140,"RP23-182D20.3, RP23-328J8.2, RP23-88B16.7, RP2..."
2,395,xref_id,Xref ID,ID of associated database reference,48543,"OTTMUSG00000012817, OTTMUSG00000059803, OTTMUS..."
3,538,legacy_biotype,Legacy biotype,Obsolete biotype previously assigned to this E...,32889,"antisense, lincRNA, sense_intronic, processed_..."
4,54,remark,Remark,Annotation remark,23109,"TAGENE_gene, SCYGR8, this gene object does not..."
5,380,havana_cv,Havana CV term,Controlled vocabulary terms from Havana,11661,"overlapping locus, readthrough gene, overlaps ..."
6,382,NoTransRefError,No translations due to reference error,This gene is believed to include protein codin...,5,1
7,1,embl_acc,European Nucleotide Archive (was EMBL) accession,ENA,0,None
8,2,status,Status,None,0,None
9,3,synonym,Synonym,None,0,None


## genes

In [8]:
ensg.gene_df.head()

/home/runner/work/ensembl-genes/ensembl-genes/ensembl_genes/ensembl_genes.py:141: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(self._alt_allele_add_representative)


,ensembl_gene_id,ensembl_gene_version,gene_symbol,gene_symbol_source_db,gene_symbol_source_id,gene_biotype,ensembl_source,ensembl_created_date,ensembl_modified_date,coord_system_version,...,seq_region_start,seq_region_end,seq_region_strand,primary_assembly,lrg_gene_id,mhc,gene_description,gene_description_source_db,gene_description_source_id,ensembl_representative_gene_id
0,ENSMUSG00000000001,6,Gnai3,MGI,MGI:95773,protein_coding,ensembl_havana,2004-11-18 15:05:57,2025-08-06 16:46:17,GRCm39,...,108014590,108053469,-1,True,NaN,None,G protein subunit alpha i3,MGI Symbol,MGI:95773,ENSMUSG00000000001
1,ENSMUSG00000000003,17,Pbsn,MGI,MGI:1860484,protein_coding,ensembl_havana,2006-06-28 13:29:12,2025-08-06 16:46:17,GRCm39,...,76881506,76897229,-1,True,NaN,None,probasin,MGI Symbol,MGI:1860484,ENSMUSG00000000003
2,ENSMUSG00000000028,17,Cdc45,MGI,MGI:1338073,protein_coding,ensembl_havana,2007-06-26 09:54:29,2025-08-06 16:46:17,GRCm39,...,18599195,18631551,-1,True,NaN,None,cell division cycle 45,MGI Symbol,MGI:1338073,ENSMUSG00000000028
3,ENSMUSG00000000031,20,H19,MGI,MGI:95891,lncRNA,ensembl_havana,2006-02-02 11:27:21,2024-05-06 19:10:14,GRCm39,...,142129262,142133957,-1,True,NaN,None,"H19, imprinted maternally expressed transcript",MGI Symbol,MGI:95891,ENSMUSG00000000031
4,ENSMUSG00000000037,19,Scml2,MGI,MGI:1340042,protein_coding,ensembl_havana,2006-08-18 16:16:32,2025-08-06 16:46:17,GRCm39,...,159865513,160041214,1,True,NaN,None,Scm polycomb group protein like 2,MGI Symbol,MGI:1340042,ENSMUSG00000000037


In [9]:
# clone-based genes no longer get a symbol and are filled with the stable ID
# https://www.ensembl.info/2021/03/15/retirement-of-clone-based-gene-names/
ensg.gene_df.query("gene_symbol == ensembl_gene_id").head(2)

,ensembl_gene_id,ensembl_gene_version,gene_symbol,gene_symbol_source_db,gene_symbol_source_id,gene_biotype,ensembl_source,ensembl_created_date,ensembl_modified_date,coord_system_version,...,seq_region_start,seq_region_end,seq_region_strand,primary_assembly,lrg_gene_id,mhc,gene_description,gene_description_source_db,gene_description_source_id,ensembl_representative_gene_id
20415,ENSMUSG00000074196,12,ENSMUSG00000074196,None,None,transcribed_unprocessed_pseudogene,havana,2008-12-18 20:15:00,2022-04-05 22:25:40,GRCm39,...,144567587,144595506,-1,True,NaN,None,"chloride channel calcium activated 4C, pseudogene",NaN,NaN,ENSMUSG00000074196
20607,ENSMUSG00000074720,2,ENSMUSG00000074720,None,None,protein_coding,ensembl,2020-04-27 16:31:13,2020-08-25 18:37:19,GRCm39,...,25923,27230,-1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000074720


In [10]:
# which external database the gene symbol derives from versus the ensembl source
pd.crosstab(
    ensg.gene_df.ensembl_source,
    ensg.gene_df.gene_symbol_source_db.fillna("missing (clone-based)"),
    margins=True,
)

gene_symbol_source_db,EntrezGene,HGNC,MGI,missing (clone-based),All
ensembl_source,,,,,
RefSeq,0,0,37,0,37
ensembl,3,0,4920,51,4974
ensembl_havana,31,0,22457,2,22490
havana,62,2,26713,153,26930
havana_tagene,63,0,22322,345,22730
mirbase,0,0,1187,0,1187
All,159,2,77636,551,78348


In [11]:
ensg.gene_df.coord_system.value_counts().head(10)

coord_system
chromosome    78289
scaffold         59
Name: count, dtype: int64

In [12]:
ensg.gene_df.gene_biotype.value_counts().head(10)

gene_biotype
lncRNA                                32889
protein_coding                        21818
processed_pseudogene                   9312
TEC                                    3219
miRNA                                  2206
unprocessed_pseudogene                 2172
snoRNA                                 1507
snRNA                                  1381
transcribed_unprocessed_pseudogene      978
transcribed_processed_pseudogene        938
Name: count, dtype: int64

In [13]:
pd.crosstab(ensg.gene_df.coord_system, ensg.gene_df.primary_assembly, margins=True)

primary_assembly,False,True,All
coord_system,,,
chromosome,0,78289,78289
scaffold,59,0,59
All,59,78289,78348


In [14]:
ensg.gene_df.mhc.value_counts()

Series([], Name: count, dtype: int64)

In [15]:
len(ensg.gene_df)

78348

## alternative gene alleles

Related:

- [OTP: Origin of genes_with_non_reference_ensembl_ids.tsv](https://github.com/opentargets/platform/issues/702)
- [biostars: map between different assemblies of one ensembl release](https://www.biostars.org/p/143956/)
- using `attrib_type.code = "non_ref"` for `primary_assembly` doesn't appear to return any results

In [16]:
ensg.representative_gene_df.head()

,rs_allele_group,ensembl_gene_id,gene_symbol,ensembl_created_date,seq_region,primary_assembly,alt_allele_group_id,alt_allele_attrib,alt_allele_is_representative,ensembl_representative_gene_id,is_representative_gene
0,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik,2007-12-11 14:58:31,7,True,None,None,False,ENSMUSG00000109644,True
1,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik,2015-07-06 11:47:18,7,True,None,None,False,ENSMUSG00000108652,True
2,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik,2005-06-29 11:35:32,2,True,None,None,False,ENSMUSG00000086714,True
3,0610009K14Rik,ENSMUSG00000142821,0610009K14Rik,2024-05-06 19:10:14,4,True,None,None,False,ENSMUSG00000142821,True
4,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik,2004-08-20 10:27:42,11,True,None,None,False,ENSMUSG00000043644,True


In [17]:
# looks like non_ref isn't set for human genes
query = """
SELECT *
FROM gene_attrib
LEFT JOIN attrib_type
  ON gene_attrib.attrib_type_id = attrib_type.attrib_type_id
WHERE attrib_type.code = "non_ref"
LIMIT 5
"""
pd.read_sql(sql=query, con=ensg.connection_url)

,gene_id,attrib_type_id,value,attrib_type_id,code,name,description


In [18]:
ensg.representative_gene_df.alt_allele_attrib.value_counts()

Series([], Name: count, dtype: int64)

In [19]:
ensg.representative_gene_df.query(
    "ensembl_gene_id != ensembl_representative_gene_id"
).head(2)

,rs_allele_group,ensembl_gene_id,gene_symbol,ensembl_created_date,seq_region,primary_assembly,alt_allele_group_id,alt_allele_attrib,alt_allele_is_representative,ensembl_representative_gene_id,is_representative_gene
62,1500026H17Rik,ENSMUSG00000144250,1500026H17Rik,2024-08-28 15:22:29,10,True,None,None,False,ENSMUSG00000097383,False
80,1600029O15Rik,ENSMUSG00000121691,1600029O15Rik,2023-02-25 19:58:01,9,True,None,None,False,ENSMUSG00000057818,False


# replaced ID converter

A single `old_stable_id` can map to multiple `new_stable_id`. For example, `ENSG00000152006`

https://uswest.ensembl.org/Homo_sapiens/Tools/IDMapper/Results?tl=AzhM62SpkvdiLC4H-6808613

Requested ID | Matched ID(s) | Releases
-- | -- | --
ENSG00000152006 | ENSG00000196273 | 26: ENSG00000196273.1
ENSG00000152006 | ENSG00000197016 | 26: ENSG00000197016.1
ENSG00000152006 | ENSG00000196239 | 26: ENSG00000196239.1

In [20]:
ensg.old_to_new_df.head(2)

,old_ensembl_gene_id,new_ensembl_gene_id
0,ENSMUSG00000037155,ENSMUSG00000044465
1,ENSMUSG00000000700,ENSMUSG00000066358


In [21]:
# some ensembl genes replaced by many new ensembl genes
ensg.old_to_new_df.old_ensembl_gene_id.value_counts().head(2)

old_ensembl_gene_id
ENSMUSG00000070230    131
ENSMUSG00000070062    131
Name: count, dtype: int64

In [22]:
# example
ensg._update_ensembl_gene("ENSG00000152006")

{'ENSG00000152006'}

In [23]:
ensg.old_to_newest_df.head(2)

,old_ensembl_gene_id,newest_ensembl_gene_id,is_current
0,ENSMUSG00000000700,ENSMUSG00000066358,False
1,ENSMUSG00000000700,ENSMUSG00000068941,False


In [24]:
len(ensg.old_to_newest_df)

71744

In [25]:
ensg.old_to_newest_df.is_current.value_counts()

is_current
True     58900
False    12844
Name: count, dtype: int64

## omni-updater

The omni-updater dataset is designed to convert ensembl gene IDs from input data to the current, representative ensembl_gene_ids for this ensembl release. It assumes:

- users want to update outdated genes with their replacements
- users want a dataset of representative genes only, and want to convert alternative alleles to representative genes

An inner join of a dataset with `update_df` on `input_ensembl_gene_id` will do the following:

- produce output ensembl_gene_ids that are current and representatives
- update outdated genes with their current identifiers. Outdated genes with no current replacement will be removed by the inner join.
- update alternative gene alleles with their representatives
- genes that are already represenative and current will map to themselves

In [26]:
ensg.update_df.head(2)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene
0,ENSMUSG00000000001,ENSMUSG00000000001,True,True,1,1
1,ENSMUSG00000000003,ENSMUSG00000000003,True,True,1,1


In [27]:
ensg.update_df.sort_values("input_maps_to_n_genes", ascending=False).head(2)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene
14215,ENSMUSG00000075786,ENSMUSG00000119739,False,True,126,255
14216,ENSMUSG00000075786,ENSMUSG00000119740,False,True,126,255


In [28]:
ensg.update_df.sort_values("n_inputs_map_to_gene", ascending=False).head(2)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene
19408,ENSMUSG00000075867,ENSMUSG00000119539,False,True,120,255
19410,ENSMUSG00000075867,ENSMUSG00000119576,False,True,120,255


In [29]:
(ensg.update_df.input_maps_to_n_genes == 1).mean()

np.float64(0.5806277687106551)

In [30]:
ensg.update_df.query("ensembl_gene_id == 'ENSG00000256263'")

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene


In [31]:
print(
    f"The omni-updater contains {len(ensg.update_df):,} rows for mapping "
    f"{ensg.update_df.input_ensembl_gene_id.nunique():,} input genes to "
    f"{ensg.update_df.ensembl_gene_id.nunique():,} current, representative genes."
)

The omni-updater contains 137,248 rows for mapping 82,598 input genes to 78,045 current, representative genes.


In [32]:
# https://useast.ensembl.org/Homo_sapiens/Tools/IDMapper/Results?tl=P45VLMbogubpI0QA-6815464
ensg.update_df.query("input_ensembl_gene_id == 'ENSG00000201456'").head(3)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene


## cross-refrences (xrefs)

In [33]:
ensg.xref_df.head()

,ensembl_representative_gene_id,ensembl_gene_id,gene_symbol,xref_source,xref_accession,xref_label,xref_description,xref_info_type,xref_linkage_annotation,xref_curie
0,ENSMUSG00000000001,ENSMUSG00000000001,Gnai3,ArrayExpress,ENSMUSG00000000001,ENSMUSG00000000001,None,DIRECT,None,arrayexpress:ENSMUSG00000000001
1,ENSMUSG00000000001,ENSMUSG00000000001,Gnai3,EntrezGene,14679,Gnai3,G protein subunit alpha i3,DEPENDENT,None,ncbigene:14679
2,ENSMUSG00000000001,ENSMUSG00000000001,Gnai3,MGI,MGI:95773,Gnai3,G protein subunit alpha i3,DIRECT,None,mgi:95773
3,ENSMUSG00000000001,ENSMUSG00000000001,Gnai3,Reactome_gene,R-MMU-109582,R-MMU-109582,Hemostasis,DIRECT,None,reactome:R-MMU-109582
4,ENSMUSG00000000001,ENSMUSG00000000001,Gnai3,Reactome_gene,R-MMU-111885,R-MMU-111885,Opioid Signalling,DIRECT,None,reactome:R-MMU-111885


In [34]:
# datasets where there are ensembl_gene_id-xref_source-xref_accession pairs might not be distinct
xref_dup_df = ensg.xref_df[
    ensg.xref_df.duplicated(
        subset=["ensembl_gene_id", "xref_source", "xref_accession"], keep=False
    )
]
xref_dup_df.xref_source.value_counts()

Series([], Name: count, dtype: int64)

In [35]:
# xref sources versus info_types
df = pd.crosstab(ensg.xref_df.xref_source, ensg.xref_df.xref_info_type, margins=True)
df["bioregistry_prefix"] = (
    df.index.to_series().replace(ensg._xref_prefix_updater).map(normalize_prefix)
)
df

xref_info_type,DEPENDENT,DIRECT,MISC,PROJECTION,All,bioregistry_prefix
xref_source,,,,,,
ArrayExpress,0,78348,0,0,78348,arrayexpress
EntrezGene,28681,0,0,0,28681,ncbigene
HGNC,0,0,0,2,2,hgnc
MGI,0,77636,0,0,77636,mgi
RFAM,0,2467,0,0,2467,rfam
Reactome_gene,0,90227,0,0,90227,reactome
Uniprot_gn,51912,0,0,0,51912,uniprot
WikiGene,28681,0,0,0,28681,wikigenes
miRBase,0,0,1360,0,1360,mirbase


## Gene Ontology xrefs

In [36]:
ensg.xref_go_df.head(3)

,ensembl_gene_id,go_id,go_label,go_evidence_codes,xref_info_types,xref_info_texts,ensembl_transcript_ids,ensembl_representative_gene_id
0,ENSMUSG00000000001,GO:0000139,Golgi membrane,"IEA,ISO","PROJECTION,DIRECT","from homo_sapiens translation ENSP00000358867,...",ENSMUST00000000001,ENSMUSG00000000001
1,ENSMUSG00000000001,GO:0000166,nucleotide binding,IEA,DIRECT,UniProt,ENSMUST00000000001,ENSMUSG00000000001
2,ENSMUSG00000000001,GO:0001664,G protein-coupled receptor binding,IBA,DIRECT,GO_Central,ENSMUST00000000001,ENSMUSG00000000001


In [37]:
# GO terms for CCR5
# compare to http://useast.ensembl.org/Homo_sapiens/Gene/Ontologies/molecular_function?g=ENSG00000160791
# Using sort_values() to handle potential None values in go_label
ensg.xref_go_df.query(
    "ensembl_gene_id == 'ENSG00000160791'"
).go_label.sort_values().tolist()

[]

In [38]:
# Check for GO terms with missing labels
# https://github.com/related-sciences/ensembl-genes/issues/28
missing_go_label_df = ensg.xref_go_df[ensg.xref_go_df.go_label.isna()]
if not missing_go_label_df.empty:
    print(f"Found {len(missing_go_label_df):,} GO annotations with missing go_label")
    display(missing_go_label_df.head(10))
else:
    print("All GO terms have labels")

Found 15,856 GO annotations with missing go_label


,ensembl_gene_id,go_id,go_label,go_evidence_codes,xref_info_types,xref_info_texts,ensembl_transcript_ids,ensembl_representative_gene_id
57,ENSMUSG00000000028,GO:1902977,None,IBA,DIRECT,GO_Central,ENSMUST00000000028,ENSMUSG00000000028
233,ENSMUSG00000000093,GO:0036302,None,IMP,DIRECT,BHF-UCL,ENSMUST00000000095,ENSMUSG00000000093
259,ENSMUSG00000000093,GO:1901208,None,IMP,DIRECT,BHF-UCL,ENSMUST00000000095,ENSMUSG00000000093
260,ENSMUSG00000000093,GO:1901211,None,IMP,DIRECT,BHF-UCL,ENSMUST00000000095,ENSMUSG00000000093
262,ENSMUSG00000000093,GO:1905072,None,"IEA,ISO","PROJECTION,DIRECT","from homo_sapiens translation ENSP00000240328,...",ENSMUST00000000095,ENSMUSG00000000093
263,ENSMUSG00000000093,GO:1905222,None,"IEA,IGI",DIRECT,"BHF-UCL,UniProt",ENSMUST00000000095,ENSMUSG00000000093
265,ENSMUSG00000000093,GO:2000773,None,"IEA,ISO","PROJECTION,DIRECT","from homo_sapiens translation ENSP00000240328,...",ENSMUST00000000095,ENSMUSG00000000093
296,ENSMUSG00000000103,GO:0051598,None,IDA,DIRECT,MGI,"ENSMUST00000115891,ENSMUST00000187148",ENSMUSG00000000103
330,ENSMUSG00000000120,GO:0031069,None,IMP,DIRECT,MGI,ENSMUST00000000122,ENSMUSG00000000120
374,ENSMUSG00000000120,GO:1902895,None,"IEA,ISO","PROJECTION,DIRECT","from homo_sapiens translation ENSP00000172229,...",ENSMUST00000000122,ENSMUSG00000000120


## lrg xrefs

In [39]:
ensg.xref_lrg_df.head(2)

,ensembl_gene_id,lrg_gene_id


In [40]:
len(ensg.xref_lrg_df)

0

### ncbigene xrefs

In [41]:
ensg.xref_ncbigene_df.head()

,ensembl_representative_gene_id,ncbigene_id,gene_symbol,ncbigene_symbol
1,ENSMUSG00000000001,14679,Gnai3,Gnai3
26,ENSMUSG00000000003,54192,Pbsn,Pbsn
33,ENSMUSG00000000028,12544,Cdc45,Cdc45
51,ENSMUSG00000000031,14955,H19,H19
55,ENSMUSG00000000037,107815,Scml2,Scml2


In [42]:
# ensembl gene mapped to by multiple ncbigenes
ensg.xref_ncbigene_df.ensembl_representative_gene_id.value_counts().head(3)

ensembl_representative_gene_id
ENSMUSG00000000001    1
ENSMUSG00000000003    1
ENSMUSG00000000028    1
Name: count, dtype: int64

In [43]:
(
    len(ensg.xref_ncbigene_df),
    ensg.xref_ncbigene_df.ensembl_representative_gene_id.duplicated().sum(),
)

(28292, np.int64(0))

In [44]:
# ncbigene mapped to by multiple ensembl genes, likely due to alt gene alleles
ensg.xref_ncbigene_df.ncbigene_id.value_counts().head(3)

ncbigene_id
108168560    7
102635519    4
207728       2
Name: count, dtype: int64

In [45]:
len(ensg.xref_ncbigene_df), ensg.xref_ncbigene_df.ncbigene_id.duplicated().sum()

(28292, np.int64(28))

In [46]:
# ensg.xref_ncbigene_df.query("ensembl_representative_gene_id == 'ENSG00000231500'")
# ensg.xref_ncbigene_df.query("ncbigene_id == '51206'")

In [47]:
repr_ensembl_gene_ids = set(ensg.gene_df.ensembl_representative_gene_id)
len(repr_ensembl_gene_ids)

78045

In [48]:
# many of these genes should probably be alternative alleles rather than representative
ensg.gene_df.query(
    "not primary_assembly and ensembl_gene_id==ensembl_representative_gene_id"
)

,ensembl_gene_id,ensembl_gene_version,gene_symbol,gene_symbol_source_db,gene_symbol_source_id,gene_biotype,ensembl_source,ensembl_created_date,ensembl_modified_date,coord_system_version,...,seq_region_start,seq_region_end,seq_region_strand,primary_assembly,lrg_gene_id,mhc,gene_description,gene_description_source_db,gene_description_source_id,ensembl_representative_gene_id
17491,ENSMUSG00000062783,3,Csprs,EntrezGene,114564,protein_coding,ensembl,2020-04-27 16:28:04,2020-08-25 18:37:28,GRCm39,...,111572,163011,-1,False,NaN,None,component of Sp100-rs,NCBI gene (formerly Entrezgene),114564,ENSMUSG00000062783
20607,ENSMUSG00000074720,2,ENSMUSG00000074720,None,None,protein_coding,ensembl,2020-04-27 16:31:13,2020-08-25 18:37:19,GRCm39,...,25923,27230,-1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000074720
22206,ENSMUSG00000079190,4,ENSMUSG00000079190,None,None,protein_coding,ensembl,2020-04-27 16:17:40,2020-08-25 18:37:25,GRCm39,...,167446,196478,1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000079190
22207,ENSMUSG00000079192,3,ENSMUSG00000079192,None,None,protein_coding,ensembl,2020-04-27 16:11:51,2020-08-25 18:37:27,GRCm39,...,123792,124928,1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000079192
22210,ENSMUSG00000079222,3,ENSMUSG00000079222,None,None,protein_coding,ensembl,2020-04-27 16:26:11,2020-08-25 18:37:27,GRCm39,...,17591,30203,-1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000079222
22432,ENSMUSG00000079794,4,ENSMUSG00000079794,None,None,protein_coding,ensembl,2020-04-27 16:13:32,2025-08-06 16:46:17,GRCm39,...,129081,142770,-1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000079794
22433,ENSMUSG00000079800,4,ENSMUSG00000079800,None,None,protein_coding,ensembl,2020-04-27 16:09:17,2025-08-06 16:46:17,GRCm39,...,9124,59053,-1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000079800
22435,ENSMUSG00000079808,4,ENSMUSG00000079808,None,None,protein_coding,ensembl,2020-04-27 16:05:55,2020-08-25 18:37:18,GRCm39,...,128555,150452,-1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000079808
30055,ENSMUSG00000091585,3,ENSMUSG00000091585,None,None,protein_coding,ensembl,2020-04-27 17:39:52,2020-08-25 18:37:22,GRCm39,...,716296,720517,1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000091585
31215,ENSMUSG00000094054,2,ENSMUSG00000094054,None,None,processed_pseudogene,ensembl,2020-04-27 16:15:58,2020-08-25 18:37:25,GRCm39,...,113868,114758,1,False,NaN,None,NaN,NaN,NaN,ENSMUSG00000094054
